# Python Practice Generator

In-kernel pandas, numpy, and basic-Python drills for a Data Analyst III interview. No Docker and no database: every problem builds a small in-memory DataFrame, array, or list right in the kernel.

**Workflow:** pick a topic, difficulty, and (optionally) a scenario, then generate a problem. Write your answer in the code cell (assign it to `result`), then click Check. Show Solution reveals an idiomatic reference.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">\u25B2 Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Setup ──
import os, sys
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd
import numpy as np

# Reload the engine so edits to the .py take effect on re-run
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
if 'nb02_python_drill_utils' in sys.modules:
    del sys.modules['nb02_python_drill_utils']
import nb02_python_drill_utils as pdu

# Shared state across cells
STATE = {'problem': None, 'hint_index': 0}

print('Engine ready. Categories:', len(pdu.category_keys()))

# Keep Shift+Enter / Cmd+Z native inside the answer textarea so re-running the
# cell does not wipe widget state and native undo keeps working.
from IPython.display import Javascript
display(Javascript("""
  document.addEventListener('keydown', function(e) {
    var inField = (e.target.tagName === 'TEXTAREA' || e.target.tagName === 'INPUT');
    if (!inField) return;
    if (e.shiftKey && e.key === 'Enter') { e.preventDefault(); e.stopPropagation(); }
    var modifier = e.metaKey || e.ctrlKey;
    if (modifier && (e.key === 'z' || e.key === 'Z')) { e.stopPropagation(); }
  }, true);
"""))

# Auto-grow the answer textarea as you type.
display(Javascript("""
  (function() {
    function autoResize(ta){var st=window.scrollY; ta.style.height='auto'; ta.style.height=(ta.scrollHeight+2)+'px'; window.scrollTo({top:st});}
    function attach(ta){if(ta.dataset.arAttached) return; ta.dataset.arAttached='1'; ta.style.overflow='hidden'; ta.addEventListener('input',function(){autoResize(ta);}); setTimeout(function(){autoResize(ta);},50);}
    function poll(){document.querySelectorAll('.py-code-editor textarea').forEach(attach);}
    poll(); setInterval(poll, 1000);
  })();
"""))


## 1. Pick a problem

Choose a Topic, Difficulty, and an optional Scenario flavor, then Generate. Source New builds a fresh random problem; Replay last re-shows the current one unchanged.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">\u25B2 Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Problem Picker ──

# Topic dropdown, grouped by family in the label so the three sections read clearly.
_topic_opts = []
for _k in pdu.category_keys():
    _m = pdu.CATEGORIES[_k]
    _topic_opts.append((f"[{_m['group']}] {_m['label']}", _k))
topic_dd = widgets.Dropdown(options=_topic_opts, value=pdu.category_keys()[0],
    description='Topic:', style={'description_width': '90px'},
    layout=widgets.Layout(width='460px'))

difficulty_radio = widgets.RadioButtons(
    options=[('\U0001F7E2 Easy','easy'),('\U0001F7E1 Moderate','moderate'),('\U0001F534 Hard','hard')],
    value='easy', description='Difficulty:', style={'description_width':'90px'},
    layout=widgets.Layout(width='420px'))

scenario_dd = widgets.Dropdown(options=[
        ('\U0001F3B2 Random (any scenario)','random'),
        ('Consumer Social','consumer_social'),
        ('Marketplace','marketplace'),
        ('Ecommerce','ecommerce'),
        ('Fintech','fintech'),
        ('B2B SaaS','b2b_saas'),
        ('Productivity & Media','productivity_media'),
        ('Health & Wellness','health_wellness'),
        ('Gaming','gaming'),
        ('Education','education'),
        ('Pharmacy & Care','pharmacy_care'),
    ], value='random', description='Scenario:', style={'description_width':'90px'},
    layout=widgets.Layout(width='420px'))

source_radio = widgets.RadioButtons(
    options=[('New (generate)','new'),('Replay last','replay')], value='new',
    description='Source:', style={'description_width':'90px'},
    layout=widgets.Layout(width='320px'))

generate_btn = widgets.Button(description='Generate Problem', button_style='primary',
    layout=widgets.Layout(width='200px', height='34px'))
status_out = widgets.Output()
problem_out = widgets.Output()

def render_problem(p):
    with problem_out:
        clear_output(wait=True)
        display(HTML(pdu.render_problem_html(p)))

def on_generate(b):
    with status_out:
        clear_output(wait=True)
        if source_radio.value == 'replay' and STATE.get('problem'):
            p = STATE['problem']
            print('Replaying current problem (unchanged).')
        else:
            p = pdu.generate_problem(topic_dd.value, difficulty_radio.value, scenario=scenario_dd.value)
            STATE['problem'] = p
            STATE['hint_index'] = 0
            print(f'Generated: [{p.group}] {p.label} ({p.difficulty})')
            print(f'Scenario: {p.scenario}')
            print(f"Assign your answer to `{p.result_var}` in the cell below.")
        try:
            refresh_reminder()
        except NameError:
            pass
        render_problem(p)

generate_btn.on_click(on_generate)

display(widgets.VBox([
    topic_dd, difficulty_radio, scenario_dd, source_radio,
    generate_btn, status_out, problem_out,
]))


## 2. Write your answer

The problem card is repeated below for reference. Write pandas / numpy / Python in the editor and assign your result to the variable named in the prompt (usually `result`). Then click Check.

- **Run code**: execute your code and show what `result` holds (no grading).
- **Check**: compare `result` to the expected output, pass or fail.
- **Hint**: reveal the next hint.
- **Show Solution**: reveal the reference solution.

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">\u25B2 Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Answer Editor + Checker ──

def _render_problem_reminder():
    p = STATE.get('problem')
    if not p:
        return widgets.HTML('<div style="color:#57606a; padding:10px;"><i>Generate a problem first.</i></div>')
    return widgets.HTML(pdu.render_problem_html(p))

reminder_box = widgets.VBox([_render_problem_reminder()])
def refresh_reminder():
    reminder_box.children = [_render_problem_reminder()]

code_ta = widgets.Textarea(value='# write your answer here; assign it to `result`\n\nresult = None\n',
    placeholder='# pandas / numpy / python', layout=widgets.Layout(width='100%', min_height='200px'))
code_ta.add_class('py-code-editor')

run_btn = widgets.Button(description='Run code', button_style='', layout=widgets.Layout(width='120px'))
check_btn = widgets.Button(description='Check', button_style='success', layout=widgets.Layout(width='120px'))
hint_btn = widgets.Button(description='Hint', button_style='warning', layout=widgets.Layout(width='120px'))
sol_btn  = widgets.Button(description='Show Solution', button_style='info', layout=widgets.Layout(width='150px'))
result_out = widgets.Output()
hint_out = widgets.Output()

def _build_namespace(p):
    ns = {'pd': pd, 'np': np}
    for k, v in p.inputs.items():
        ns[k] = v.copy() if hasattr(v, 'copy') else v
    return ns

def _run_user_code(p):
    ns = _build_namespace(p)
    exec(p.setup_code, ns)  # rebuild inputs from setup so names always exist
    for k, v in p.inputs.items():
        ns.setdefault(k, v)
    exec(code_ta.value, ns)
    return ns.get(p.result_var, None)

def on_run(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.'); return
        try:
            val = _run_user_code(p)
        except Exception as e:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{e}</pre></div>')); return
        display(HTML(f'<h4>Your <code>{p.result_var}</code></h4>' + pdu._obj_to_html(val)))

def on_check(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.'); return
        try:
            val = _run_user_code(p)
        except Exception as e:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error running your code:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{e}</pre></div>')); return
        ok, msg = pdu.check_answer(p, val)
        color = '#dcfce7' if ok else '#ffebe9'
        bar = '#1a7f37' if ok else '#cf222e'
        verdict = 'PASS' if ok else 'FAIL'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        if not ok:
            display(HTML('<h4>Your result</h4>' + pdu._obj_to_html(val)))
            display(HTML('<h4>Expected</h4>' + pdu._obj_to_html(p.expected)))

def on_hint(b):
    with hint_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.'); return
        hints = p.hints or ['No hints for this one; try Show Solution.']
        idx = STATE.get('hint_index', 0)
        text = hints[min(idx, len(hints)-1)]
        STATE['hint_index'] = min(idx+1, len(hints))
        display(HTML(f'<div style="background:#fff8c5; border-left:4px solid #d4a72c; padding:10px 14px; border-radius:4px;"><strong>Hint {min(idx+1,len(hints))}/{len(hints)}:</strong> {text}</div>'))

def on_solution(b):
    with hint_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.'); return
        display(HTML(pdu.render_solution_html(p)))

run_btn.on_click(on_run)
check_btn.on_click(on_check)
hint_btn.on_click(on_hint)
sol_btn.on_click(on_solution)

display(widgets.VBox([reminder_box, code_ta,
    widgets.HBox([run_btn, check_btn, hint_btn, sol_btn]), hint_out, result_out]))

# Dark editor styling for the answer textarea.
display(HTML("""
<style>
.py-code-editor textarea {
  font: 14px/1.5 ui-monospace, Consolas, Menlo, monospace !important;
  tab-size: 4; -moz-tab-size: 4;
  background: #282a36 !important; color: #f8f8f2 !important;
  border: 1px solid #44475a !important; border-radius: 6px;
  padding: 10px 12px !important;
}
</style>
"""))


## 3. Next problem

Clear the editor and pick a fresh problem from Section 1 (set Source to New and Generate again).

In [ ]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">\u25B2 Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Reset ──
next_btn = widgets.Button(description='Clear editor', button_style='danger',
    layout=widgets.Layout(width='180px', height='34px'))
next_out = widgets.Output()
def on_next(b):
    code_ta.value = '# write your answer here; assign it to `result`\n\nresult = None\n'
    STATE['hint_index'] = 0
    with hint_out:
        clear_output()
    with result_out:
        clear_output()
    with next_out:
        clear_output(wait=True)
        print('Editor cleared. Generate a new problem in Section 1.')
next_btn.on_click(on_next)
display(widgets.VBox([next_btn, next_out]))


---

**No Docker, no database.** Every input object is built in the kernel by `nb02_python_drill_utils.generate_problem`. The checker (`pdu.check_answer`) compares your `result` to the expected output across DataFrame, Series, ndarray, scalar, list, and dict.